In [2]:
! pip install PyMuPDF==1.26.1

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [1]:
! which python

/home/jovyan/.mlspace/envs/ekolodin-embeddings/bin/python


# Анализ и подготовка документов

На этом занятии рассмотрим, как с помощью разных библиотек в Python можно обрабатывать PDF-документы, извлекать необходимую информацию и дальше работать с ней, как с текстом.

# Часть 1. Базовая обработка документов/текста

**Компоненты:**  
* PDF-документ на выбор  
* Модель для создания эмбеддингов на выбор  

**Шаги:**  
1. Импортировать PDF-документ.
2. Обработать текст для эмбеддингов (например, разбить на предложения/фрагменты). Текст разбивается на логические части (чанки) для эффективного моделирования.
3. Создать эмбеддинги для текстовых фрагментов с помощью выбранной модели. Используются современные модели (например, BERT, Sentence-BERT, GPT-Embeddings).  
4. Сохранить эмбеддинги в файл для последующего использования  
*(Эмбеддинги будут храниться годами или до потери жесткого диска.)* Рекомендуемые форматы хранения: `.npy`, `.h5`, `.pt` (PyTorch).

## Импорт PDF-документов  
Этот подход работает с разными типами документов. Мы начнем с PDF из-за популярности формата.  

**Важно:** метод также поддерживает:  
- Текстовые файлы (.txt)  
- Цепочки писем (email threads)  
- Техническую документацию  
- Статьи и научные работы  

**Пример использования:**  
Представим, что мы студенты-диетологи Гавайского университета, изучающие открытый учебник [*«Питание человека: издание 2020 года»*](https://pressbooks.oer.hawaii.edu/humannutrition2/).  

In [2]:
import os

pdf_path = "Глубокое обучение.pdf"

assert os.path.exists(pdf_path), f"Файл {pdf_path} не найден"
print(f"Файл {pdf_path} найден.")

Файл Глубокое обучение.pdf найден.


**PDF успешно загружен!**

Мы можем преобразовать страницы PDF в текстовый формат, выполнив шаги:  
1. Определить путь к PDF-файлу.  
2. Открыть и прочитать его с помощью библиотеки **PyMuPDF** (используется через `import fitz`).

**Процесс обработки:**  
- Создайте **вспомогательную функцию** для предобработки текста во время чтения.  
- Обратите внимание: текст может считываться неоднородно (например, разное форматирование таблиц/текста), что требует адаптивной обработки.  

**Структура данных:**  
Каждая страница сохраняется в виде словаря, который затем добавляется в список для удобства дальнейшего использования.  

Пример структуры словаря:  
```python
{
    "page_number": 1,
    "text": "Содержание страницы...",
    "metadata": {"автор": "Иван Петров"}
}
```

In [3]:
# Требуется установка PyMuPDF: !pip install PyMuPDF (репозиторий: https://github.com/pymupdf/pymupdf)
# Примечание: библиотека использует лицензию AGPL-3.0 (учитывайте при коммерческом использовании кода)
import fitz  # pymupdf (выбрана вместо pypdf для нашего кейса)
from tqdm.auto import tqdm  # индикатор выполнения, установка: !pip install tqdm


def text_formatter(text: str) -> str:
    """Выполняет базовое форматирование текста."""
    cleaned_text = text.replace("\n", " ").strip()  # примечание: обработка может отличаться для разных документов
    return cleaned_text  # сюда можно добавить другие функции форматирования


# Чтение PDF и извлечение текста/страниц
# Примечание: работает только с текстом (изображения/графики игнорируются)
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Открывает PDF-файл и извлекает текст со страниц со сбором статистики.

    Параметры:
        pdf_path (str): Путь к PDF-документу

    Возвращает:
        list[dict]: Список словарей с данными для каждой страницы:
            - page_number: номер страницы (с коррекцией)
            - page_char_count: количество символов
            - page_word_count: количество слов
            - page_sentence_count_raw: примерное количество предложений
            - page_token_count: оценка токенов (1 токен ≈ 4 символа)
            - text: обработанный текст страницы
    """
    doc = fitz.open(pdf_path)  # открываем документ
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # итерируем страницы с прогресс-баром
        text = page.get_text()  # получаем текст в UTF-8
        text = text_formatter(text)
        pages_and_texts.append({
            "page_number": page_number - 22,  # коррекция нумерации (PDF начинается с 42 стр.)
            "page_char_count": len(text),
            "page_word_count": len(text.split(" ")),
            "page_sentence_count_raw": len(text.split(". ")),
            "page_token_count": len(text) // 4,  # расчет токенов: https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them
            "text": text
        })
    return pages_and_texts


pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]  # просмотр первых двух элементов

0it [00:00, ?it/s]

[{'page_number': -22,
  'page_char_count': 2582,
  'page_word_count': 331,
  'page_sentence_count_raw': 20,
  'page_token_count': 645,
  'text': 'Интернет\ue02dмагазин: www.dmkpress.com Книга – почтой: e\ue02dmail: orders@alians-kniga.ru Оптовая продажа:  «Альянс\ue02dкнига» Тел./факс: (499) 782\ue02d3889 e\ue02dmail: books@alians-kniga.ru www.дмк.рф Глубокое обучение — это вид машинного обучения, наделяющий компьютеры  способностью учиться на опыте. Книга содержит математические и концептуальные  основы линейной алгебры, теории вероятностей и теории информации, численных  расчетов и машинного обучения в том объеме, который необходим для понимания  материала. Описываются приемы глубокого обучения, применяемые на практике, в том  числе глубокие сети прямого распространения, регуляризация, алгоритмы оптимизации,  сверточные сети, моделирование последовательностей и др. Рассматриваются такие  приложения, как обработка естественных языков, распознавание речи, компьютерное  зрение, онлайнов

Теперь возьмем случайную выборку страниц.

In [4]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 43,
  'page_char_count': 2485,
  'page_word_count': 368,
  'page_sentence_count_raw': 26,
  'page_token_count': 621,
  'text': 'Условная вероятность\u2003 \uf076\u2003 65 \x81 \x81 Область определения p\xa0– множество всех возможных состояний x. \x81 \x81 ∀x ∈ x, p(x) ≥ 0; отметим, что мы не требуем выполнения условия p(x) ≤ 1. \x81 \x81 ∫p(x)dx = 1. Функция плотности вероятности p(x) определяет не вероятность конкретного со- стояния, а\xa0вероятность попадания в\xa0бесконечно малую окрестность размера δx, кото- рая равна p(x)δx. Для нахождения  вероятности множества точек следует проинтегрировать функ- цию плотности. Точнее, вероятность, что x принадлежит множеству 𝕊, равна инте- гралу p(x) по этому множеству. В\xa0одномерном случае вероятность принадлежности x  отрезку [a, b] равна ∫[a,\u2009b] p(x)dx. В качестве примера рассмотрим равномерное распределение непрерывной случай- ной величины на отрезке вещественных чисел. Для этого нужно определить функции  u(x; a, b),

## Получение статистики по текстам

Проведем предварительный анализ данных (EDA), чтобы понять объем текста, количество символов, слов и так далее. Это поможет определить оптимальный способ их разбиения.

Многие модели для создания эмбеддингов имеют ограничения на длину входных текстов. Например, модель [`all-mpnet-base-v2`](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) из библиотеки [`sentence-transformers`](https://www.sbert.net/docs/pretrained_models.html) принимает максимум **384 токена**.
  
Это означает, что модель обучена работать с текстами длиной до 384 токенов (где: 1 токен ≈ 4 символа ≈ 0.75 слова).

Тексты длиннее 384 токенов будут автоматически обрезаться при обработке этой моделью, что может привести к потере информации.

Более подробно мы обсудим это в разделе про создание эмбеддингов.

А сейчас преобразуем наш список словарей в `DataFrame` и проанализируем его.

In [5]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-22,2582,331,20,645,Интернетмагазин: www.dmkpress.com Книга – поч...
1,-21,61,8,1,15,"Ян Гудфеллоу, Иошуа Бенджио, Аарон Курвилль Гл..."
2,-20,115,15,1,28,"Ian Goodfellow, Yoshua Bengio, Aaron Courville..."
3,-19,112,15,1,28,"Ян Гудфеллоу, Иошуа Бенджио, Аарон Курвилль Гл..."
4,-18,1916,254,18,479,"УДК 004.85 ББК 32.971.3 Г93 Гудфеллоу Я., Бенд..."


In [6]:
# Статистика
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,653.00,653.00,653.00,653.00,653.00
mean,304.00,2759.54,364.75,26.27,689.49
std,188.65,728.07,93.57,21.45,182.02
min,-22.00,61.00,8.00,1.00,15.00
25%,141.00,2324.00,307.00,17.00,581.00
50%,304.00,2850.00,375.00,22.00,712.00
75%,467.00,3321.00,440.00,26.00,830.00
max,630.00,5307.00,528.00,120.00,1326.00


По этим данным среднее количество токенов на страницу составляет 287.

Это означает, что для нашего случая можно использовать модель `all-mpnet-base-v2` (с максимальной длиной входа 384 токена) для создания эмбеддингов целых страниц без обрезки.

## Дальнейшая обработка текста (разбиение страниц на предложения)  
Идеальный способ предобработки текста перед созданием эмбеддингов все еще является предметом исследований.  

**Простой рабочий метод:** разбивать текст на группы предложений (чанки) по 5, 7, 10 или более предложений. Эти значения можно настраивать под ваши задачи.  



### Общий рабочий процесс:  
1. Загрузить текст.  
2. Разделить на чанки.  
3. Создать эмбеддинги для чанков.  
4. Использовать эмбеддинги.



### Методы разбиения на предложения:  
- **Базовый:** Разделение по `. ` (например, `text.split(". ")`, как делали ранее).
- **Продвинутый:** Использование NLP-библиотек (spaCy, nltk).  



### Зачем разбивать на предложения?  
- Упрощение обработки (особенно для плотных текстов).  
- Возможность точно определить, **какие предложения** повлияли на результат в RAG-пайплайне.



**Рекомендация:** используйте spaCy для разбиения, так как это надежнее, чем простое разделение по `. `.

In [7]:
# Импорт английского языка для spaCy (инструкции по установке: https://spacy.io/usage)
from spacy.lang.en import English

# Создаем объект NLP для английского языка
nlp = English()

# Добавляем компонент sentencizer в пайплайн (документация: https://spacy.io/api/sentencizer/)
nlp.add_pipe("sentencizer")

# Создаем пример документа для тестирования
doc = nlp("This is a sentence. This is another sentence.")
assert len(list(doc.sents)) == 2  # Проверяем, что текст разбит на 2 предложения

# Получаем и выводим список предложений документа
list(doc.sents)

[This is a sentence., This is another sentence.]

Необязательно использовать spaCy, однако это библиотека с открытым исходным кодом, специально созданная для масштабной обработки NLP-задач, таких как наша.  

Давайте применим наш небольшой **пайплайн для разбиения на предложения** к текстовым страницам.

In [8]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)

    # Проверяем что все предложения -- это строки
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    # Считаем предложения
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/653 [00:00<?, ?it/s]

In [9]:
# Проверим
random.sample(pages_and_texts, k=1)

[{'page_number': 390,
  'page_char_count': 2103,
  'page_word_count': 309,
  'page_sentence_count_raw': 14,
  'page_token_count': 525,
  'text': '412\u2003 \uf076\u2003 Линейные факторные модели  x = Wh + b + noise Рис. 13.1\u2002 \uf076\u2002 Ориентированная графическая модель, описывающая се- мейство линейных факторных моделей. Предполагается, что наблюдаемый  вектор x является линейной комбинацией независимых латентных факто- ров h и\xa0 шума. В\xa0 различных моделях, например в\xa0 вероятностном методе  главных компонент, факторном анализе и\xa0анализе независимых компонент  (ICA), делаются различные предположения о\xa0 форме шума и\xa0 априорном  распределении p(h) 13.1. Вероятностный PCA и факторный анализ В факторном анализе (Bartholomew, 1987; Basilevsky, 1994) латентная переменная  имеет априорное нормальное распределение с\xa0единичной дисперсией h ∼ 𝒩(h; 0, I),\t (13.3) а наблюдаемые переменные xi предполагаются условно независимыми при условии  h. Точнее говоря, предполагае

Отлично! Теперь преобразуем наш список словарей в `DataFrame` и получим статистические данные.

In [10]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,653.00,653.00,653.00,653.00,653.00,653.00
mean,304.00,2759.54,364.75,26.27,689.49,25.45
std,188.65,728.07,93.57,21.45,182.02,16.37
min,-22.00,61.00,8.00,1.00,15.00,1.00
25%,141.00,2324.00,307.00,17.00,581.00,17.00
50%,304.00,2850.00,375.00,22.00,712.00,23.00
75%,467.00,3321.00,440.00,26.00,830.00,27.00
max,630.00,5307.00,528.00,120.00,1326.00,96.00


Похоже, что для нашего набора текстов грубый подсчет предложений (например, разделение по `". "`) довольно близок к тому, что показал spaCy.

Теперь, когда у нас есть текст, разделенный на предложения, давайте сгруппируем эти предложения.

## Группировка предложений

Давайте сделаем шаг к разбиению нашего списка предложений или текста на мелкие части.

Как вы могли догадаться, этот процесс называется **чанкинг** (группировка).

### Почему мы это делаем?

1. Легче управлять частями текста похожего размера.
2. Не перегружать емкость моделей для создания эмбеддингов токенами (например, если модель для создания эмбеддингов имеет емкость 384 токена, может произойти потеря информации, если вы попытаетесь создать эмбеддинг для последовательности из 400+ токенов).
3. Наше окно контекста LLM (количество токенов, которое LLM может обработать) может быть ограничено и требует вычислительной мощности, поэтому мы хотим убедиться, что используем его максимально эффективно.

**На заметку**: существует множество способов для создания частей текста.

Пока мы будем держаться простого подхода и разобьем наши страницы с предложениями на группы по 10 (это число произвольное и может быть изменено, я выбрал его потому, что оно хорошо соответствует емкости нашей модели для создания эмбеддингов в 384 токена).

В среднем каждая из наших страниц содержит 10 предложений.

И в среднем 287 токенов на страницу.

Таким образом, наши группы по 10 предложений также будут длиной около 287 токенов.

Это дает нам достаточно места для создания эмбеддингов текста с помощью нашей модели `all-mpnet-base-v2` (емкость модели — 384 токена).

Чтобы разделить наши группы предложений на части по 10 или меньше, давайте создадим функцию, которая принимает список на вход и рекурсивно разбивает его на подсписки указанного размера.

In [11]:
# Определим размер разделения для преобразования групп предложений в чанки
num_sentence_chunk_size = 10

# Создадим функцию, которая рекурсивно разбивает список на нужные размеры
def split_list(input_list: list,
               slice_size: int) -> list[list[str]]:
    """
    Разбивает input_list на подсписки размером slice_size (или максимально близко к этому).

    Например, список из 17 предложений будет разбит на два списка [[10], [7]]
    """
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Перебираем страницы и тексты и разбиваем предложения на чанки
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/653 [00:00<?, ?it/s]

In [12]:
# Возьмем пример из группы (примечание: многие примеры имеют только 1 чанк, так как у них <=10 предложений всего)
random.sample(pages_and_texts, k=1)

[{'page_number': 85,
  'page_char_count': 2907,
  'page_word_count': 387,
  'page_sentence_count_raw': 21,
  'page_token_count': 726,
  'text': 'Емкость, переобучение и\xa0недообучение\u2003 \uf076\u2003 107 Эти факторы соответствуют двум центральным проблемам машинного обучения:  недообучению и\xa0переобучению. Недообучение имеет место, когда модель не позво- ляет получить достаточно малую ошибку на обучающем наборе, а\xa0переобучение\xa0–  когда разрыв между ошибками обучения и\xa0тестирования слишком велик. Управлять склонностью модели к\xa0переобучению или недообучению позволяет ее  емкость (capacity). Неформально говоря, емкость модели описывает ее способность  к\xa0аппроксимации широкого спектра функций. Модели малой емкости испытывают  сложности в\xa0аппроксимации обучающего набора. Модели большой емкости склонны  к\xa0переобучению, поскольку запоминают свойства обучающего набора, не присущие  тестовому. Один из способов контроля над емкостью алгоритма обучения состоит в\xa0том,

In [13]:
# Создадим DataFrame для получения статистики
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,653.00,653.00,653.00,653.00,653.00,653.00,653.00
mean,304.00,2759.54,364.75,26.27,689.49,25.45,2.99
std,188.65,728.07,93.57,21.45,182.02,16.37,1.65
min,-22.00,61.00,8.00,1.00,15.00,1.00,1.00
25%,141.00,2324.00,307.00,17.00,581.00,17.00,2.00
50%,304.00,2850.00,375.00,22.00,712.00,23.00,3.00
75%,467.00,3321.00,440.00,26.00,830.00,27.00,3.00
max,630.00,5307.00,528.00,120.00,1326.00,96.00,10.00


Обратите внимание, что среднее количество чанков составляет около 1.5. Это ожидаемо, так как многие из наших страниц содержат в среднем только 10 предложений.

## Разделение каждого чанка на отдельный элемент

Мы хотим преобразовать каждый чанк предложений в его собственное числовое представление.

Чтобы сохранить порядок, давайте создадим новый список словарей, где каждый словарь будет содержать один чанк предложений вместе с соответствующей информацией (такой как номер страницы), а также статистику о каждом чанке.

In [14]:
import re

# Разделяем каждый чанк на отдельный элемент
pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]

        # Объединяем предложения в структуру абзаца, т.е. чанк (таким образом, они являются одной строкой)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk) # ".A" -> ". A" для любой комбинации точки и заглавной буквы
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # Получаем статистику о чанке
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # 1 token = ~4 characters

        pages_and_chunks.append(chunk_dict)

# Сколько у нас чанков?
len(pages_and_chunks)

  0%|          | 0/653 [00:00<?, ?it/s]

1955

In [15]:
# Посмотрим на случайный образец
random.sample(pages_and_chunks, k=1)

[{'page_number': 308,
  'sentence_chunk': '330\u2003 \uf076\u2003 Моделирование последовательностей: рекуррентные и рекурсивные сети  при условии переменных в\xa0момент t стационарно, т.\xa0е.соотношение между состоянием системы в\xa0предыдущий и\xa0в\xa0последующий моменты не зависит от t. В\xa0принципе, можно было бы считать t дополнительным входом на каждом временном шаге и\xa0позволить обучаемой модели выявить временные зависимости между различными шагами.Это было бы гораздо лучше, чем использовать разные условные распределения для каж- дого t, но тогда сеть должна была бы выполнять экстраполяцию на новые значения t. Чтобы завершить рассмотрение РНС как графической модели, мы должны еще опи- сать, как производить выборку из модели.Основная интересующая нас операция\xa0– выборка примера из условного распределения на каждом временном шаге.Однако имеется одно дополнительное осложнение.У\xa0РНС должен быть какой-то механизм определения длины последовательности.Достичь этого можно разны

Отлично!

Теперь мы разбили весь наш учебник на чанки по 10 предложений или меньше, а также сохранили номер страницы, из которой они взяты.

Это означает, что мы можем ссылаться на чанк текста и знать его источник.

Давайте получим статистику по нашим чанкам.

In [16]:
# Получаем статистику о наших чанках
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1955.00,1955.00,1955.00,1955.00
mean,339.21,909.30,110.12,227.32
std,201.94,478.75,58.90,119.69
min,-22.00,4.00,1.00,1.00
25%,162.50,451.00,58.00,112.75
50%,346.00,970.00,114.00,242.50
75%,538.00,1258.00,153.00,314.50
max,630.00,3310.00,451.00,827.50


Хм, похоже, что некоторые из наших чанков имеют довольно низкое количество токенов.

Давайте проверим примеры с менее чем 30 токенами (примерно длина предложения) и посмотрим, стоит ли их сохранять.

In [17]:
# Покажем случайные чанки с длиной менее 30 токенов
min_token_length = 100
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

Chunk token count: 89.25 | Text: Анализ медленных признаков...........................................................................................415 13.4.Разреженное кодирование..................................................................................................417 13.5.Интерпретация PCA в терминах многообразий.........................................................419
Chunk token count: 91.5 | Text: 691.	 Solomonoff, R. J. (1989). A system for incremental learning based on algorithmic probability.692.	 Sontag, E. D. (1998). VC dimension of neural networks. NATO ASI Series F Com- puter and Systems Sciences, 168, 69–96.693.	 Sontag, E. D. and Sussman, H. J. (1989). Backpropagation can give rise to spu­ rious local minima even for networks without hidden layers.
Chunk token count: 16.75 | Text: Рассмотрим производную по одному из сме- щений: 	 (19.21) 	 (19.22)
Chunk token count: 61.25 | Text: Вместо оптимизации риска напрямую мы оптимизируем эмпирический риск и надеем

Похоже, что многие из них являются заголовками и подписями разных страниц.

И они не содержат много полезной информации.

Давайте отфильтруем наш DataFrame/список словарей, чтобы включить только чанки с длиной более 30 токенов.

In [18]:
pages_and_chunks_over_min_token_len = df[(df["chunk_token_count"] > min_token_length) & (df["page_number"] > 0)].to_dict(orient="records")

pages_and_chunks_over_min_token_len[:2]

[{'page_number': 1,
  'sentence_chunk': 'Стохастическая максимизация правдоподобия\u2003 \uf076\u2003 23 быстрее, если коллекция структурирована и\xa0 подходящим образом индексирована. Люди же легко выполняют арифметические операции с\xa0числами, записанными араб- скими цифрами, но тратят куда больше времени, если используются римские цифры. Неудивительно, что выбор представления оказывает огромное влияние на качество и\xa0производительность алгоритмов машинного обучения.На рис.1.1 приведен прос\xad той наглядный пример.Многие задачи ИИ можно решить, если правильно подобрать признаки, а\xa0затем предъявить их алгоритму машинного обучения.Например, в\xa0задаче идентифика- ции говорящего по звукам речи полезным признаком является речевой тракт.Он позволяет с\xa0большой точностью определить, кто говорит: мужчина, женщина или ребенок.Но во многих задачах нелегко понять, какие признаки выделять.Допустим, к\xa0при- меру, что мы пишем программу обнаружения автомобилей на фотографиях.',
  'chu

Меньшие чанки отфильтрованы!

Пришло время создать эмбеддинги для наших чанков текста.

# Часть 2. Продвинутая предобработка PDF-документов

Для начала установим все нужные зависимости.

In [27]:
!pip -q install pdfminer.six==20231228
!pip -q install pikepdf
!pip -q install matplotlib==3.10.0
!pip -q install unstructured-inference==1.0.5
!pip -q install unstructured-pytesseract==0.3.15
!pip -q install unstructured==0.17.2

In [23]:
%%bash
apt-get update -qq
apt-get install -y -qq tesseract-ocr-rus libtesseract-dev poppler-utils

In [24]:
!pip install -q nltk==3.9.1
!pip install -q pi_heif==0.22.0
!pip install -q pdf2image==1.17.0

### Разделение PDF на сущности

In [73]:
import os
import shutil

if shutil.which("tesseract") is None:
    os.environ["TESSDATA_PREFIX"] = "/home/jovyan/miniconda3/share/tessdata"

from unstructured.partition.pdf import partition_pdf

In [78]:
report_path = "Глубокое обучение.pdf"

In [79]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [80]:
report_raw_data = partition_pdf(
    filename=report_path,
    strategy="fast",
    languages=["rus"],
    extract_images_in_pdf=True,
    extract_image_block_to_payload=False,
    extract_image_block_output_dir="./data/images/"
)

Этот код выполняет извлечение данных из PDF-документа с использованием функции `partition_pdf`. Давайте разберем его по частям:

1. **`partition_pdf`**: функция `partition_pdf` предназначена для разделения PDF-документа на отдельные части, такие как текст, изображения и другие элементы. Она позволяет получить структурированные данные из PDF-файла.

2. **`filename=report_path`**: здесь передается путь к PDF-документу, который нужно обработать. Переменная `report_path` содержит этот путь.

3. **`strategy="hi_res"`**: параметр `strategy` указывает на стратегию обработки PDF. Значение `"hi_res"` говорит о том, что PDF будет обрабатываться с высоким разрешением. Это может быть полезно, если в документе содержатся сложные элементы или мелкий текст.

4. **`languages=["rus"]`**: указывает, что для извлечения текста из PDF будет использоваться русский язык. Это важно для корректного распознавания текста в документах на русском языке.

5. **`extract_images_in_pdf=True`**: параметр задает извлечение изображений из PDF. Если он установлен в `True`, то изображения, найденные в документе, будут извлекаться.

6. **`extract_image_block_to_payload=False`**: определяет, будут ли извлеченные изображения включены в результат в виде данных (payload). Поскольку значение `False`, изображения не будут включены в основной результат в виде бинарных данных. Вместо этого они будут сохранены в указанной директории.

7. **`extract_image_block_output_dir="./data/images/"`**: указывает путь к директории, куда будут сохраняться извлеченные изображения. Здесь изображения сохраняются в папку `./data/images/`.

Для других файлов могут понадобиться дополнительные параметры, которые можно найти в [документации](https://docs.unstructured.io/open-source/core-functionality/partitioning#partition-pdf).

### Извлечение текста

In [81]:
from unstructured.documents.elements import NarrativeText

In [82]:
def extract_text_with_metadata(esg_report, source_document):
    # Инициализация пустого списка для хранения текста с метаданными
    text_data = []

    # Инициализация словаря для подсчета параграфов на каждой странице
    paragraph_counters = {}

    # Проход по всем элементам в esg_report
    for element in esg_report:
        # Проверка, является ли элемент текстовым блоком (объектом NarrativeText)
        if isinstance(element, NarrativeText):
            # Извлечение номера страницы из метаданных элемента
            page_number = element.metadata.page_number

            # Если на этой странице еще нет параграфов, инициализируем счетчик параграфов
            if page_number not in paragraph_counters:
                paragraph_counters[page_number] = 1
            else:
                # Если параграфы на странице уже есть, увеличиваем счетчик
                paragraph_counters[page_number] += 1

            # Определение текущего номера параграфа на странице
            paragraph_number = paragraph_counters[page_number]

            # Извлечение текста из элемента
            text_content = element.text

            # Добавление текста и его метаданных в список text_data
            text_data.append({
                "source_document": source_document,  # Название или путь к исходному документу
                "page_number": page_number,          # Номер страницы, на которой находится текст
                "paragraph_number": paragraph_number, # Номер параграфа на текущей странице
                "text": text_content                 # Сам текст
            })

    # Возврат списка с текстом и соответствующими метаданными
    return text_data

In [83]:
extracted_data = extract_text_with_metadata(report_raw_data, report_path)

Проверим, что получилось.

In [84]:
extracted_data

[{'source_document': 'Глубокое обучение.pdf',
  'page_number': 1,
  'paragraph_number': 1,
  'text': 'Глубокое обучение — это вид машинного обучения, наделяющий компьютеры способностью учиться на опыте. Книга содержит математические и концептуальные основы линейной алгебры, теории вероятностей и теории информации, численных расчетов и машинного обучения в том объеме, который необходим для понимания материала. Описываются приемы глубокого обучения, применяемые на практике, в том числе глубокие сети прямого распространения, регуляризация, алгоритмы оптимизации, сверточные сети, моделирование последовательностей и др. Рассматриваются такие приложения, как обработка естественных языков, распознавание речи, компьютерное зрение, онлайновые рекомендательные системы, биоинформатика и видеоигры. Наконец, описываются перспективные направления исследований: линейные факторные модели, автокодировщики, обучение представлений, структурные вероятностные модели, методы Монте-Карло, статистическая сумм

### Извлечение изображений

In [85]:
from unstructured.documents.elements import Image

In [86]:
def extract_image_metadata(esg_report, source_document):
    # Инициализация пустого списка для хранения метаданных изображений
    image_data = []

    # Проход по всем элементам в esg_report
    for element in esg_report:
        # Проверка, является ли элемент изображением (объектом Image)
        if isinstance(element, Image):
            # Извлечение номера страницы из метаданных элемента
            page_number = element.metadata.page_number

            # Извлечение пути к изображению из метаданных элемента (если он существует)
            image_path = element.metadata.image_path if hasattr(element.metadata, 'image_path') else None

            # Добавление метаданных изображения в список image_data
            image_data.append({
                "source_document": source_document,  # Название или путь к исходному документу
                "page_number": page_number,          # Номер страницы, на которой находится изображение
                "image_path": image_path             # Путь к изображению (если доступен)
            })

    # Возврат списка с метаданными изображений
    return image_data

In [87]:
extracted_image_data = extract_image_metadata(report_raw_data, report_path)

In [88]:
extracted_image_data

[]

In [89]:
import matplotlib.pyplot as plt
from PIL import Image
import math

In [90]:
def display_images_from_metadata(extracted_image_data, images_per_row=4):
    # Фильтрация данных, чтобы оставить только те, где есть путь к изображению
    valid_images = [img for img in extracted_image_data if img['image_path']]

    # Если нет допустимых изображений, выводим сообщение и прекращаем выполнение
    if not valid_images:
        print("No valid image data available.")
        return

    # Определяем количество изображений и количество строк для отображения
    num_images = len(valid_images)
    num_rows = math.ceil(num_images / images_per_row)

    # Создание субплотов для отображения изображений
    fig, axes = plt.subplots(num_rows, images_per_row, figsize=(20, 5*num_rows))

    # Преобразуем массив осей в одномерный, если несколько строк, иначе делаем список из одного элемента
    axes = axes.flatten() if num_rows > 1 else [axes]

    # Цикл для отображения каждого изображения
    for ax, img_data in zip(axes, valid_images):
        try:
            # Открытие изображения по пути
            img = Image.open(img_data['image_path'])
            # Отображение изображения на соответствующей оси
            ax.imshow(img)
            ax.axis('off')  # Отключаем оси для улучшения внешнего вида
            ax.set_title(f"Page {img_data['page_number']}", fontsize=10)  # Устанавливаем заголовок с номером страницы
        except Exception as e:
            # Если произошла ошибка при загрузке изображения, выводим сообщение об ошибке
            print(f"Error loading image {img_data['image_path']}: {str(e)}")
            ax.text(0.5, 0.5, f"Error loading image\n{str(e)}", ha='center', va='center')
            ax.axis('off')

    # Удаление пустых осей, если изображений меньше, чем количество доступных осей
    for ax in axes[num_images:]:
        fig.delaxes(ax)

    # Улучшение компоновки изображения и отображение графика
    plt.tight_layout()
    plt.show()

Убедимся, что все изображения извлечены корректно.

In [91]:
display_images_from_metadata(extracted_image_data)

No valid image data available.


### Извлечение таблиц

Также можем извлекать и таблицы из pdf документов:

In [92]:
from unstructured.documents.elements import Table

In [93]:
def extract_table_metadata(esg_report, source_document):
    # Инициализация пустого списка для хранения метаданных таблиц
    table_data = []

    # Проход по всем элементам в esg_report
    for element in esg_report:
        # Проверка, является ли элемент таблицей (объектом Table)
        if isinstance(element, Table):
            # Извлечение номера страницы из метаданных элемента
            page_number = element.metadata.page_number

            # Преобразование таблицы в строковое представление
            table_content = str(element)

            # Добавление метаданных таблицы в список table_data
            table_data.append({
                "source_document": source_document,  # Название или путь к исходному документу
                "page_number": page_number,          # Номер страницы, на которой находится таблица
                "table_content": table_content       # Содержимое таблицы в виде строки
            })

    # Возврат списка с метаданными таблиц
    return table_data

In [94]:
extracted_table_data = extract_table_metadata(report_raw_data, report_path)

Проверим, что все таблицы извлечены корректно.

In [95]:
extracted_table_data

[]

Ваша задача:

**Проделать все манипуляции по извлечению сущностей для данных вашего проекта и продемонстрировать примеры работы указанных функций.**

# Часть 3. Обработка PDF-документов с помощью LangChain

В этой части рассмотрим примеры загрузки и обработки PDF-документов с помощью библиотеки LangChain.

LangChain предоставляет несколько классов-«загрузчиков» (loaders) для извлечения текста из PDF-файлов. К основным из них относятся: `PyPDFLoader`, `PDFMinerLoader`, `PDFPlumberLoader`, `PyMuPDFLoader`, `UnstructuredPDFLoader` и другие. Каждый из них использует свой парсер (`PyPDF`, `PDFMiner`, `PDFPlumber`, `PyMuPDF`, `Unstructured` и т.д.) и обладает разными возможностями.

На примерах разберем, как с их помощью загрузить PDF, получить текст и сравнить результаты. По умолчанию эти загрузчики возвращают список объектов `Document`, каждый из которых содержит текст (`page_content`) и метаданные (`metadata`), например номер страницы и путь к файлу.

In [96]:
!pip install -q langchain-community pdfplumber pymupdf

In [97]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PDFMinerLoader,
    PDFPlumberLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader
)

## Пример 1: загрузка PDF с помощью `PyPDFLoader`

`PyPDFLoader` — простой загрузчик на основе библиотеки `pypdf` (ранее PyPDF2). Он извлекает текст в том виде, в котором он хранится внутри PDF (без OCR), и возвращает один объект `Document` на страницу. Например:

In [98]:
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"Всего страниц: {len(docs)}")
print("Текст первой страницы:")
print(docs[0].page_content[:200])

Всего страниц: 653
Текст первой страницы:
Интернетмагазин:
www.dmkpress.com
Книга – почтой:
email: orders@alians-kniga.ru
Оптовая продажа: 
«Альянскнига»
Тел./факс: (499) 7823889
email: books@alians-kniga.ru
www.дмк.рф
Г лубокое обучение


В документации `LangChain` сказано, что `PyPDFLoader` возвращает список объектов `Document`, по одному на страницу, а содержание страницы доступно через атрибут `page_content`. Метаданные каждого документа (доступные в `docs[i].metadata`) содержат, например, номер страницы и путь к файлу.

## Пример 2: загрузка с помощью `PDFPlumberLoader`

`PDFPlumberLoader` основан на библиотеке `pdfplumber`. Как и `PyPDFLoader`, он возвращает по одному документу на страницу, но дополнительно сохраняет расширенные метаданные (которые извлекает `PyMuPDF` внутри себя). Документация говорит, что результирующие документы содержат подробные метаданные о PDF и его страницах и возвращают по одному документу на страницу. Использование:

In [99]:
loader = PDFPlumberLoader(pdf_path)
docs_plumber = loader.load()
print(f"Всего страниц (plumber): {len(docs_plumber)}")
print("Метаданные первой страницы:")
print(docs_plumber[0].metadata)
print("Текст первой страницы (plumber):")
print(docs_plumber[0].page_content[:200])

Всего страниц (plumber): 653
Метаданные первой страницы:
{'source': 'Глубокое обучение.pdf', 'file_path': 'Глубокое обучение.pdf', 'page': 0, 'total_pages': 653, 'CreationDate': "D:20171102225920+04'00'", 'Creator': 'Adobe InDesign CC 2017 (Windows)', 'ModDate': "D:20171108152833+03'00'", 'Producer': 'Adobe PDF Library 15.0', 'Trapped': 'False'}
Текст первой страницы (plumber):
Глубокое обучение — это вид машинного обучения, наделяющий компьютеры
способностью учиться на опыте. Книга содержит математические и концептуальные
основы линейной алгебры, теории вероятностей и теори


Сравнивая выводы, можно заметить, что формат текста и переносы строк могут немного отличаться от `PyPDFLoader`, но общая информация та же (один документ на страницу). Метаданные содержат поля типа `Author`, `Producer`, `CreationDate` и т.п.

## Пример 3: загрузка с помощью `PyMuPDFLoader`

`PyMuPDFLoader` использует библиотеку `PyMuPDF` (MuPDF). По умолчанию он тоже выдает один документ на страницу и дополнительно умеет извлекать изображения и таблицы (в режиме `load`, а также поддерживает ленивую загрузку). Инициализация и загрузка аналогичны:

In [100]:
loader = PyMuPDFLoader(pdf_path)
docs_mu = loader.load()
print(f"Всего страниц (PyMuPDF): {len(docs_mu)}")
print("Метаданные первой страницы:")
print(docs_mu[0].metadata)
print("Текст первой страницы (PyMuPDF):")
print(docs_mu[0].page_content[:200])

Всего страниц (PyMuPDF): 653
Метаданные первой страницы:
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 2017 (Windows)', 'creationdate': '2017-11-02T22:59:20+04:00', 'source': 'Глубокое обучение.pdf', 'file_path': 'Глубокое обучение.pdf', 'total_pages': 653, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2017-11-08T15:28:33+03:00', 'trapped': '', 'modDate': "D:20171108152833+03'00'", 'creationDate': "D:20171102225920+04'00'", 'page': 0}
Текст первой страницы (PyMuPDF):
Интернетмагазин:
www.dmkpress.com
Книга – почтой:
email: orders@alians-kniga.ru
Оптовая продажа: 
«Альянскнига»
Тел./факс: (499) 7823889
email: books@alians-kniga.ru
www.дмк.рф
Глубокое обучение 


Как и у `PyPDFLoader`, результатом является список документов. Документация `PyMuPDFLoader` отмечает возможности извлечения изображений и таблиц, но в базовой загрузке он просто дает текст страницы.

## Пример 4: загрузка с помощью `PDFMinerLoader`

`PDFMinerLoader` основан на библиотеке `pdfminer.six`. Он поддерживает два режима: "single" (весь документ как один текст) и "page" (по странице на документ). По умолчанию mode='single' и тогда весь текст возвращается как один объект `Document`, а страницы в нем разделены специальным разделителем (по умолчанию `pages_delimiter="\n\x0c"` — символ перевода страницы). Пример:

In [101]:
# Загрузка как один документ
loader = PDFMinerLoader(pdf_path, mode="single")
docs_miner_single = loader.load()
print(f"Документов (PDFMiner single): {len(docs_miner_single)}")
print("Текст (начало):")
print(docs_miner_single[0].page_content[:200])
# Загрузка постранично
loader_page = PDFMinerLoader(pdf_path, mode="page")
docs_miner_page = loader_page.load()
print(f"Всего страниц (PDFMiner): {len(docs_miner_page)}")

Документов (PDFMiner single): 1
Текст (начало):
Глубокое  обучение  —  это  вид  машинного  обучения,  наделяющий  компьютеры 
способностью  учиться  на  опыте.  Книга  содержит  математические  и  концептуальные 
основы  линейной  алгебры,  теории
Всего страниц (PDFMiner): 653


В режиме `"page"` `PDFMinerLoader` вернет список документов, по одному на страницу (аналогично `PyPDFLoader`). В режиме `"single"` — один документ с текстом всех страниц, разделенных указанным разделителем. Можно вручную разбить такой текст на страницы по `"\f"` или другой подстроке. Это покажет, как каждый загрузчик строит документы.

## Пример 5: загрузка с помощью `UnstructuredPDFLoader`

`UnstructuredPDFLoader` — мощный загрузчик на основе библиотеки `Unstructured`, способный извлекать сложные структуры из документа (заголовки, таблицы, списки) и выполнять OCR при необходимости. Он поддерживает два режима:
* `mode="single"` — весь документ возвращается как один `Document`;
* `mode="elements"` — документ разбивается на элементы (например, заголовки, параграфы, таблицы и пр.), где каждый — отдельный `Document`.

Например, загрузим в режиме `single`:

In [103]:
loader = UnstructuredPDFLoader(report_path, mode="single", strategy="hi_res", languages=["rus"])
docs_un_single = loader.load()
print(f"Документов (Unstructured single): {len(docs_un_single)}")
print("Текст (начало):")
print(docs_un_single[0].page_content[:200])


И в режиме `elements`:

In [ ]:
loader = UnstructuredPDFLoader(report_path, mode="elements", strategy="hi_res", languages=["rus"])
docs_un_elems = loader.load()
print(f"Элементов (Unstructured elements): {len(docs_un_elems)}")
# Покажем первые несколько элементов на первой странице
elems_page1 = [doc for doc in docs_un_elems if doc.metadata.get("page_number")==1]
for i, doc in enumerate(elems_page1[:3], 1):
    print(f"Элемент {i}, тип: {doc.metadata.get('element_type')}")
    print(doc.page_content.strip()[:100])


Из документации: в режиме `elements` `UnstructuredPDFLoader` разделит документ на части типа «Title», «NarrativeText» и т.д. (в примере выше мы выводим тип элемента из метаданных). Это демонстрирует более детальную сегментацию по сравнению с простым `PyPDFLoader`.

## Сравнение загрузчиков

Каждый загрузчик по-разному разбивает и форматирует текст. Если кратко:

* **PyPDFLoader** и **PDFPlumberLoader**, **PyMuPDFLoader**: возвращают один `Document` на страницу. Текст в `page_content` — это текст страницы с переносами строк. Различия могут быть в том, как обрабатываются разрывы строк или специальные символы, однако в общем они извлекают видимый текст.
* **PDFMinerLoader**: позволяет либо вернуть всё в одном документе (`mode="single"`, страницы отделяются символом формы `\f`), либо по одному документу на страницу (`mode="page"`). Это может быть удобным, если нужен единый текст или нужно управлять разделением вручную.
* **UnstructuredPDFLoader**: по умолчанию (режим `single`) может вести себя похоже на PyPDFLoader (отдавая текст всего документа или всей страницы), но его главное преимущество — режим `elements`, который разбивает документ на структурированные элементы. Так, для сложных PDF с таблицами или заголовками он может дать более семантически осмысленные куски текста.

# Итоги
Вы научились извлекать всю необходимую информацию из сырых PDF-документов с помощью библиотек на языке Python. Эти несколько способов пригодятся вам на будущих занятиях.